In [ ]:
# ============================================================
# Experiment 1 – Behavioural Preprocessing
# ============================================================

# Description:
# Preprocessing of behavioural reaction time (RT) data
# for the implicit dual-picture AAT.
#
# Steps:
# 1. Merge session data
# 2. Convert RTs to milliseconds
# 3. Remove incorrect trials
# 4. Remove RTs < 150 ms
# 5. Winsorise upper RT tail
# 6. Export cleaned behavioural dataset
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================

RT_MIN = 150
RT_WINSOR_SD = 2

participants_to_exclude = [33, 35, 37, 54, 57, 58]

PARTICIPANTS_80_TRIALS = [1, 2, 3, 61, 62, 63, 64, 65]

DEFAULT_EXPECTED_TRIALS = 96
REDUCED_EXPECTED_TRIALS = 80

# ============================================================
# Helper Functions
# ============================================================

def load_csv(name, dtype=float):
    return pd.read_csv(name, header=None).to_numpy(dtype=dtype)

def merge_sessions(arr):
    merged = []
    for p in range(arr.shape[1] // 2):
        merged.append(np.concatenate([arr[:, 2*p], arr[:, 2*p+1]]))
    return merged

def convert_rt_to_ms(rt):
    rt = np.asarray(rt, dtype=float)

    if 0.05 <= np.nanmedian(rt) <= 5:
        return rt * 1000

    return rt.copy()

def get_intended_trial_mask(total_slots, expected_trials):

    if expected_trials == 96:
        return np.ones(total_slots, dtype=bool)

    if expected_trials == 80:
        s1 = np.r_[np.ones(40), np.zeros(8)]
        s2 = np.r_[np.ones(40), np.zeros(8)]
        return np.concatenate([s1, s2]).astype(bool)

# ============================================================
# Load Raw Data
# ============================================================

rt_raw       = load_csv("Raw_Data/reactionTime.csv")
frame_raw    = load_csv("Raw_Data/frameColor.csv")
reaction_raw = load_csv("Raw_Data/actualReaction.csv", dtype=str)

side_raw     = load_csv("Raw_Data/frameSide.csv")
valence_raw  = load_csv("Raw_Data/valence.csv")

# ============================================================
# Merge Sessions
# ============================================================

rt_parts       = merge_sessions(rt_raw)
frame_parts    = merge_sessions(frame_raw)
reaction_parts = merge_sessions(reaction_raw)

side_parts     = merge_sessions(side_raw)
valence_parts  = merge_sessions(valence_raw)

# ============================================================
# First Pass: RT Cleaning
# ============================================================

participant_data = []
global_rt_pool = []

for pid in range(1, len(rt_parts) + 1):

    if pid in participants_to_exclude:
        continue

    idx = pid - 1

    expected_trials = (
        REDUCED_EXPECTED_TRIALS
        if pid in PARTICIPANTS_80_TRIALS
        else DEFAULT_EXPECTED_TRIALS
    )

    rt = convert_rt_to_ms(rt_parts[idx])

    frame = frame_parts[idx].astype(float)

    action = np.char.lower(
        np.char.strip(reaction_parts[idx].astype(str))
    )

    intended_mask = get_intended_trial_mask(
        len(rt),
        expected_trials
    )

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    required_action = np.full(len(frame), np.nan, dtype=object)

    required_action[frame == 1] = "pull"
    required_action[frame == 0] = "push"

    correct = (
        intended_mask
        & (action == required_action)
    )

    rt_clean = rt.copy()
    rt_clean[~correct] = np.nan

    # --------------------------------------------------------
    # RT threshold
    # --------------------------------------------------------

    rt_fast_flag = (
        intended_mask
        & np.isfinite(rt_clean)
        & (rt_clean < RT_MIN)
    )

    rt_clean[rt_fast_flag] = np.nan

    global_rt_pool.extend(
        rt_clean[intended_mask & np.isfinite(rt_clean)]
    )

    participant_data.append({
        "pid": pid,
        "rt_clean": rt_clean,
        "correct": correct,
        "intended_mask": intended_mask,
        "side": side_parts[idx],
        "valence": valence_parts[idx],
        "expected_trials": expected_trials
    })

# ============================================================
# Global Winsorisation
# ============================================================

global_rt_pool = np.asarray(global_rt_pool)

rt_upper = (
    np.nanmean(global_rt_pool)
    + RT_WINSOR_SD * np.nanstd(global_rt_pool, ddof=1)
)

# ============================================================
# Final Dataset
# ============================================================

rows = []

for pdata in participant_data:

    pid = pdata["pid"]

    rt_clean = pdata["rt_clean"]
    correct = pdata["correct"]
    intended_mask = pdata["intended_mask"]

    side = pdata["side"]
    valence = pdata["valence"]

    expected_trials = pdata["expected_trials"]

    rt_clean = np.where(
        rt_clean > rt_upper,
        rt_upper,
        rt_clean
    )

    valid_trials = intended_mask & np.isfinite(rt_clean)

    trial_numbers = np.arange(1, expected_trials + 1)

    trial_counter = 0

    for t in np.where(intended_mask)[0]:

        if not valid_trials[t]:
            continue

        rows.append({

            "participant": pid,

            "trial": int(trial_numbers[trial_counter]),

            "RT_ms": float(rt_clean[t]),

            "logRT": float(np.log10(rt_clean[t])),

            "side": "right" if side[t] == 1 else "left",

            "valence": (
                "positive"
                if valence[t] == 1
                else "negative"
            )
        })

        trial_counter += 1

# ============================================================
# Save
# ============================================================

df_beh = pd.DataFrame(rows)

df_beh.to_csv(
    "Processed_Data/Exp1_behavioural_clean.csv",
    index=False
)

print("Behavioural preprocessing complete.")
print(f"Final trials: {len(df_beh)}")